In [ ]:
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
PHF       = "phash"                              # hash function name
RUN_DIR   = Path("../snapshots/phash/run_CHANGE_ME")  # path to the run folder
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_ROOT  = Path().resolve().parent
INITIAL_DIR   = PROJECT_ROOT / "problems" / PHF / "initial_programs"

In [ ]:
# Load initial programs: name -> stripped code
initial: dict[str, str] = {}
for f in sorted(INITIAL_DIR.glob("*.py")):
    initial[f.stem] = f.read_text(encoding="utf-8").strip()

print(f"Initial programs ({len(initial)}):")
for name in initial:
    print(f"  {name}")

In [ ]:
# Load evolved programs from bin_* folders: (bin, name) -> stripped code
evolved: dict[tuple[str, str], str] = {}

for bin_dir in sorted(RUN_DIR.glob("bin_*")):
    if not bin_dir.is_dir():
        continue
    for f in sorted(bin_dir.glob("*.py")):
        evolved[(bin_dir.name, f.stem)] = f.read_text(encoding="utf-8").strip()

print(f"Evolved programs across bins ({len(evolved)}):")
for (bin_name, prog_name) in evolved:
    print(f"  {bin_name}/{prog_name}")

In [ ]:
# Find exact code matches between evolved and initial programs
# Build reverse map: code -> initial name
initial_by_code = {code: name for name, code in initial.items()}

matches = []  # (bin_name, evolved_name, initial_name)
for (bin_name, prog_name), code in evolved.items():
    if code in initial_by_code:
        matches.append((bin_name, prog_name, initial_by_code[code]))

print(f"Exact matches: {len(matches)} / {len(evolved)} evolved programs\n")
if matches:
    print(f"{'Bin':<20} {'Evolved name':<35} {'Matches initial'}")
    print("-" * 75)
    for bin_name, prog_name, init_name in sorted(matches):
        print(f"{bin_name:<20} {prog_name:<35} {init_name}")
else:
    print("No exact matches found — all evolved programs differ from initials.")

In [ ]:
# Summary: which initial programs were never modified at all
matched_initials = {init_name for _, _, init_name in matches}
unmatched = set(initial) - matched_initials

print(f"Initial programs that appear verbatim in evolved set ({len(matched_initials)}):")
for name in sorted(matched_initials):
    print(f"  [=] {name}")

print(f"\nInitial programs NOT found verbatim in evolved set ({len(unmatched)}):")
for name in sorted(unmatched):
    print(f"  [-] {name}")